# PPARD UniDock Docking Audit

Objective: test whether the current PPARD UniDock setup is credible as docking and pose generation before relying on downstream rescoring.

Done for the first pass:
- prepare the 2ZNP cocrystal ligand
- check current box geometry against the crystal ligand
- build a 50-ligand audit panel
- submit/evaluate the smallest UniDock redocking run when Kaggle access is available

## Setup

This notebook is a thin driver around scripts in the same audit directory. It does not modify core SwiftScreen code.

In [ ]:
from pathlib import Path
import json

AUDIT = Path.cwd()
if AUDIT.name != "docking_audit_20260719":
    AUDIT = Path("benchmarks/bench-20260714-abl1-smoke/ppard/crystal_holo/docking_audit_20260719").resolve()
AUDIT

## Prepare Inputs

This extracts the cocrystal ligand, writes SDF/PDBQT files, snapshots box geometry, writes current top-20 pose diagnostics, and creates the small audit panel.

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, str(AUDIT / "scripts" / "prepare_audit_inputs.py")], check=True)

In [ ]:
print((AUDIT / "redocking" / "box_sanity.json").read_text())
print((AUDIT / "docking_ablation_metrics.csv").read_text())

## Smallest UniDock Redocking Run

Local UniDock is not available on this workstation. Run this cell only when a Kaggle upload/submission is allowed for the tiny cocrystal-ligand job.

In [ ]:
# subprocess.run([
#     sys.executable,
#     str(AUDIT / "scripts" / "submit_redocking_kaggle.py"),
#     "--ablation-id", "current_box_balance",
#     "--search-mode", "balance",
# ], check=True)

## Review Redocking Result

After the Kaggle redock finishes, this cell shows the latest RMSD and pose sanity diagnostics.

In [ ]:
latest = AUDIT / "redocking" / "redocking_rmsd_latest.json"
if latest.exists():
    print(latest.read_text())
else:
    print((AUDIT / "redocking" / "redocking_status.json").read_text())

## 50-Ligand Panel Baseline

This runs the tiny active/hard-decoy/neutral-decoy panel with the current PPARD receptor, current box, and UniDock balance mode. The already completed result is read in the next cell.

In [ ]:
# subprocess.run([
#     sys.executable,
#     str(AUDIT / "scripts" / "submit_panel_kaggle.py"),
#     "--ablation-id", "current_box_balance",
#     "--search-mode", "balance",
# ], check=True)

In [ ]:
panel_summary = AUDIT / "panel_runs" / "panel_summary_current_box_balance.json"
if panel_summary.exists():
    print(panel_summary.read_text())
else:
    print("Panel baseline has not been run yet.")

## Next Ablations

Add one variable at a time: box padding/center, search mode, num modes, seed stability, receptor prep variant, protonation/charge handling, and ligand conformer/prep choices.

In [ ]:
import pandas as pd

pd.read_csv(AUDIT / "audit_panel" / "audit_panel.csv").groupby(["audit_reason", "label"]).size()